# Data Governance with Unity Catalog

Unity Catalog is the unified governance layer for data and AI in Azure Databricks. It provides a single place to control access, discover assets, track lineage, classify sensitive data, and monitor quality across your organization, no matter where your data lives. Unity Catalog governs models, functions, and other AI assets as securable objects, the same way it governs your tables and volumes.

## Governance Capabilities
* **Access control:** Grant and manage fine-grained access to data and AI assets with privileges, attribute-based access control, and row filters and column masks.
* **Governed tags:** Define and control the tags used to classify and organize securable objects.
* **Data discovery:** Help users find the data and AI assets they need with Catalog Explorer, AI-generated comments, and certified and deprecated tags.
* **Data and AI lineage:** Capture and visualize how data flows through your organization, down to the column level, including the models that consume it.
* **Data classification:** Automatically scan and tag sensitive data, such as PII, across tables in Unity Catalog.
* **Data quality monitoring:** Detect anomalies across a catalog or schema and profile the statistical properties of individual tables.
* **Auditing:** Track who accessed your data and what actions they took using audit logs.
* **Data sharing:** Share data and AI assets securely across organizations and platforms with OpenSharing, Clean Rooms, and Databricks Marketplace.
* **AI governance with Unity Gateway:** Govern AI runtime interactions with Unity Gateway, including model services, coding agents, agents, AI traffic, and guardrails.

# What is Unity Catalog?

Unity Catalog is the unified governance layer for data and AI built into Azure Databricks. It operates beneath every data and AI interaction in your workspaces automatically: enforcing access control when you query a table or call a model, tracking lineage as data and AI assets are used, and logging activity for auditing. 

## The Unity Catalog Object Model

In Unity Catalog, every data and AI asset is modeled as a **securable object**. This means you can easily grant, manage, and revoke access permissions (for users, service principals, or groups) on any of these assets. 

To organize these objects logically, Unity Catalog uses a **three-level namespace** (`catalog.schema.object`) that sits beneath a top-level Metastore:

### 1. Metastore (The Top Level)
The metastore is the primary container for metadata in Unity Catalog. It sits at the very top of the hierarchy. Certain foundational objects—like storage credentials, external locations, shares, and connections—are managed directly under the metastore rather than inside a specific catalog.

### 2. The Three-Level Namespace
For your actual data and AI assets, Unity Catalog organizes them into three distinct layers:
* **Catalog (Level 1):** The first layer of the namespace. It is used to logically group and isolate data for different business units, environments (e.g., dev, test, prod), or projects.
* **Schema / Database (Level 2):** The second layer, contained within a catalog. Schemas act as folders to further organize your specific data and AI assets.
* **Object (Level 3):** The final layer, contained within a schema. These are the actual assets you interact with, which include:
  * **Data assets:** Tables, Views, and Volumes (for unstructured data).
  * **AI & Logic assets:** Functions, Machine Learning Models, Secrets, and Model Services.

### Managed vs. External Assets
When dealing with Tables and Volumes in this object model, Unity Catalog defines them in two ways based on how the underlying storage is handled:
* **Managed Assets:** Unity Catalog manages both the governance (access control) AND the underlying file storage lifecycle. If you delete a managed table, the underlying data files are also deleted.
* **External Assets:** Unity Catalog only manages the governance. The actual data files remain in your external cloud storage. If you drop an external table, Unity Catalog forgets about it, but your raw data files are safely left untouched in your storage account.

## Unity Catalog Capabilities

* **Access control:** Manage who can access what using privileges, attribute-based policies, row and column filters, and workspace bindings.
* **Discovery:** Interact with securable objects using Catalog Explorer to discover and manage data and AI assets registered in Unity Catalog.
* **Lineage:** Automatically track how data and AI assets flow and connect, from source data through to models, services, and dashboards.
* **Auditing:** Maintain a complete record of all data access and system activity using the audit log system table.
* **Data classification:** Automatically classify and tag sensitive data in your catalog.
* **Data quality monitoring:** Proactively track data health with built-in profiling and alerts that catch anomalies before they reach downstream consumers.
* **Data sharing:** Securely share live data and AI assets across organizations and clouds using the open OpenSharing protocol.
* **AI governance:** Govern AI assets and AI traffic using Unity Catalog and AI Gateway.

## The Unity Catalog Object Hierarchy

Securable objects in Unity Catalog are hierarchical, which provides the foundation for access control. The **metastore** is the top-level securable object. Within this metastore, your data assets live in a three-level namespace that defines its catalog, schema, and type of asset, such as a table (`catalog.schema.table`).

### 1. Data and AI Assets (The Three-Level Namespace)
* **Catalogs:** The top-level layer for your data assets. Catalogs exist directly under the metastore. They are used to organize your data and AI assets, typically by organizational units or software development lifecycle scopes.
* **Schemas:** Exist within catalogs. They organize data and AI assets into categories that are more granular than catalogs. A schema may represent a single use case, project, or team sandbox.
* **Tables:** Collections of structured data organized by rows and columns.
* **Views:** Saved queries against other tables or views.
* **Volumes:** Represent collections of unstructured data in cloud object storage.
* **Functions:** Units of reusable logic that return a scalar value or set of rows.
* **Models:** Versioned or unversioned AI models registered in Unity Catalog.
* **Services:** Governed, invocable AI assets, such as model services and MCP services.
* **Secrets:** Store sensitive values, such as credentials and tokens, that you can govern in Unity Catalog and use in code without exposing the value.
* **Flows:** Pipeline queries that incrementally load and process data into a target table.

### 2. External Data Source and Service Assets
These securable objects exist directly under the metastore and manage access to cloud storage and other external data sources and services:
* **Storage credentials:** Objects that represent the authentication information required to access a specific path in cloud storage.
* **External locations:** Objects that represent a specific path in cloud storage. It also includes a reference to the storage credential required to access that path.
* **External metadata:** An object used to define custom data lineage relationships for systems that operate outside of Unity Catalog.
* **Service credentials:** Objects that represent the authentication information required to access external cloud services.
* **Connections:** Objects that represent a connection to an external database system.

### 3. Data Sharing Assets
These securable objects exist directly under the metastore and manage access to data and AI asset sharing across metastore or organizational boundaries:
* **Shares:** Objects that represent a logical grouping of data assets that you intend to share with external recipients.
* **Providers:** Objects that represent an external organization or group of users that has shared data with your organization.
* **Recipients:** Objects that represent an external organization or group of users that a provider shares data with.
* **Clean rooms:** Objects that represent a secure environment for collaborating with other organizations without exposing underlying data.

## Metastore

The **metastore** is the top-level securable object in Unity Catalog. It acts as the primary container for all securable objects registered in a single cloud region. This includes not only the catalogs that organize your data, but also the foundational objects that control how data is accessed and shared (such as storage credentials, external locations, connections, and shares).

### Key Details
* **Scope:** A metastore is scoped to a single cloud region. Your organization only needs one metastore per region, and that single metastore can be attached to multiple workspaces within that region. If you grant a permission in one workspace, it automatically applies to all other workspaces sharing that same metastore.
* **Metastore Privileges:** Privileges granted on the metastore (like `CREATE CATALOG`) allow operations at the metastore level itself. Importantly, privileges granted at the metastore level **do not inherit** down to child objects. For example, being able to create a catalog does not automatically give you read access to the data inside it.
* **Metastore Admins:** This is an optional role assigned by account admins. Metastore admins have powerful capabilities that cannot be granted through standard privileges, such as deleting the metastore, managing which workspaces are attached to it, and taking ownership of any object inside it (which indirectly grants them access to all data).

## Catalog

Within a metastore, a **catalog** is the first and highest-level layer for organizing your data assets. It acts as a container for schemas (which in turn contain tables, views, volumes, and functions). In the Unity Catalog three-level namespace (`catalog.schema.table`), the catalog represents the first level.

### Key Details
* **Inheritance:** Privileges granted at the catalog level automatically cascade down to all current and future schemas, tables, views, volumes, and functions inside it. Because of this broad reach, you should be careful when granting catalog-level privileges.
* **Usage Privilege (`USE CATALOG`):** A user must have the `USE CATALOG` privilege to interact with any object inside a catalog, regardless of the permissions they hold on the individual child objects.
* **The `BROWSE` Privilege:** Granting this privilege allows users to see the metadata (like names and properties) of objects inside the catalog without granting them access to read the actual data. This is useful for data discovery.
* **Workspace Binding:** By default, a catalog can be accessed from any workspace attached to its metastore. However, you can restrict access by binding a catalog only to specific workspaces (and can even make it read-only for certain workspaces).

## Schema

Within a catalog, a **schema** (often called a database) is the second layer in the object hierarchy for your data assets. It acts as a container to further organize your specific data, holding objects like tables, views, volumes, and functions. In the Unity Catalog three-level namespace (`catalog.schema.table`), the schema represents the second level.

### Key Details
* **Inheritance:** Privileges granted at the schema level automatically cascade down to all current and future tables, views, volumes, and functions inside it. Because these privileges apply broadly to everything in the schema, you should always review what the schema contains before granting access.
* **Usage Privilege (`USE SCHEMA`):** A user must have the `USE SCHEMA` privilege to interact with any object inside that schema. This is required *in addition* to the `USE CATALOG` privilege on the parent catalog.
* **Access Limitation:** Granting `USE SCHEMA` by itself does not give a user permission to read or modify the actual data inside the schema. They still need specific object-level privileges (like `SELECT` on a table) to access the data.

## Table

Within a schema, a **table** is the primary securable object used to hold structured data (organized in rows and columns) in Unity Catalog. In the Unity Catalog three-level namespace (`catalog.schema.table`), the table represents the third and final layer.

### Types of Tables
There are three main types of tables you can use in Azure Databricks:
* **Managed tables:** Unity Catalog completely manages the storage location path for these tables. While the data itself still lives in your cloud account, Unity Catalog handles its lifecycle. Databricks recommends these to take advantage of the latest table features.
* **External tables:** You manually specify the exact storage location path for the data. Unity Catalog manages the table's metadata (for access control), but it does not manage the actual data's lifecycle, layout, or optimization.
* **Foreign tables:** These are tables that actually live in a completely different, external system or catalog, but are registered and accessed within Unity Catalog.

### Key Details
* **Usage Privileges:** To access a table, a user must first have `USE CATALOG` on its parent catalog and `USE SCHEMA` on its parent schema. Only then will their specific table-level privileges (like `SELECT` or `MODIFY`) actually allow them to see or change the data.
* **Inheritance:** You don't always have to grant permissions on the table directly. If a user is granted a privilege like `SELECT` at the schema or catalog level, they automatically inherit that same `SELECT` privilege on all current and future tables inside them.
* **Read and Write Access:** You grant `SELECT` for read access and `MODIFY` for broad write access. You can also grant finer-grained write access using `INSERT`, `UPDATE`, and `DELETE`. Note that foreign tables are strictly read-only and do not support write access.

## View

Within a schema, a **view** is a read-only object defined by a stored SQL query over one or more tables or other views. Views do not store the data themselves; instead, they recompute the results every time a query is run against them. In the Unity Catalog three-level namespace, a view sits at the third and final layer alongside tables.

### Key Details
* **Usage Privileges:** To access and query a view, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema, in addition to the `SELECT` privilege on the view itself.
* **Data Security (No Underlying Access Needed):** A user querying a view *does not* need privileges on the underlying tables that the view pulls data from. Unity Catalog uses the view owner's privileges to resolve the underlying tables at query time. This makes views an excellent tool for restricting access to specific rows or columns without exposing the raw, underlying tables directly to users.
* **Inheritance:** The `SELECT` privilege granted at the schema or catalog level automatically applies to all current and future views within that schema or catalog.

## Materialized View

A **materialized view** is a special type of view that pre-computes and actually stores its query results. The results you see reflect the state of the underlying data at the exact time the materialized view was last refreshed.

### Key Details
* **Usage Privileges:** Just like standard views, a user needs `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema (usage privileges), plus the `SELECT` privilege on the materialized view to query the stored results.
* **The `REFRESH` Privilege:** In addition to standard view privileges, materialized views support a `REFRESH` privilege. This allows a user to actively trigger an update of the materialized view's stored results.
* **Refresh Restriction:** Users who only have `SELECT` (and the required usage privileges) can query the existing stored results, but they are not allowed to trigger a new refresh.

## Metric View

A **metric view** is a read-only object that defines a set of reusable metric definitions. These definitions can be based on one or more tables, views, or SQL queries. Users interact with and query a metric view exactly as they would a standard view.

### Key Details
* **Usage Privileges:** To query a metric view, a user must have `USE CATALOG` on the parent catalog, `USE SCHEMA` on the parent schema (usage privileges), and the `SELECT` privilege on the metric view itself.
* **Data Security:** Similar to standard views, the metric view owner's privileges are used to resolve and access the underlying data sources at query time. This means the querying user does not need direct access to the raw underlying data sources.

## Flow

Within a schema, a **flow** (currently in Beta) is a securable object that represents a query within a Lakeflow pipeline. It is designed to incrementally load and process data into a target streaming table or materialized view. The pipeline that defines the flow is responsible for creating and managing it, and multiple flows can send data into a single target table.

### Key Details
* **Usage Privileges:** To access a flow, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema, in addition to a specific privilege on the flow itself.
* **Flow Privileges:** 
  * `READ FLOW`: Allows a user to describe the flow and read its current state.
  * `REFRESH FLOW`: Allows a user to trigger a refresh of the flow.
  * `WRITE FLOW`: Allows a user to reset or rewind the flow's streaming checkpoint.
  * `MANAGE`: Allows a user to manage privileges on the flow, rename it, or drop it.
* **Creation Requirements:** To create a new flow, a user must have the `CREATE FLOW` privilege on the parent schema and the `MODIFY` privilege on the target table.
* **Inheritance:** Flow privileges granted at the schema or catalog level automatically apply to all current and future flows within that schema or catalog.

## Volume

Within a schema, a **volume** is a securable object specifically designed for unstructured data in cloud storage. Unlike tables or views, volumes do not support SQL queries; instead, they provide direct file-level read and write access to your cloud storage data.

### Types of Volumes
* **Managed volumes:** Unity Catalog completely determines and manages the storage location path for these volumes. Databricks recommends these so that Unity Catalog can automatically govern all data access.
* **External volumes:** You manually specify the exact cloud storage location path. This is useful if you need external systems (outside of Databricks) to access the files, though those external systems will bypass Unity Catalog's governance.

### Key Details
* **Usage Privileges:** To access files inside a volume, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema, in addition to the `READ VOLUME` or `WRITE VOLUME` privilege on the volume itself.
* **Read and Write Access:** Granting `READ VOLUME` allows a user to read files and directories within the volume. Granting `WRITE VOLUME` allows a user to add, modify, or delete files.
* **Inheritance:** Privileges like `READ VOLUME` and `WRITE VOLUME` granted at the schema or catalog level automatically apply to all current and future volumes within that schema or catalog.

## Function

Within a schema, a **function** is a securable object in Unity Catalog that represents reusable, executable logic. 

### Types of Functions
* **User-defined functions (UDFs):** Custom functions written in SQL or Python that can be called in SQL queries and notebooks.
* **Stored procedures:** User-defined routines that execute a sequence of SQL statements and may include side effects such as inserting or updating data.
* **Registered models:** MLflow machine learning models registered in Unity Catalog. In Unity Catalog, registered models are implemented as a type of function.

### Key Details
* **Usage Privileges:** To execute a function or load a registered model, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema (usage privileges), in addition to the `EXECUTE` privilege on the function.
* **The `EXECUTE` Privilege:** Granting `EXECUTE` to a user on a function allows them to call the function and view its definition and metadata. For registered models, `EXECUTE` also allows the user to view metadata for all versions of the registered model and to download model files.
* **Inheritance:** The `EXECUTE` privilege granted at the schema or catalog level automatically applies to all current and future functions in that schema or catalog.

## Model

Within a schema, a **model** is a securable object that represents a versioned or unversioned AI model stored in Unity Catalog. It acts as a container that can hold multiple versions of a model. For example, if you use MLflow to train a model, the artifacts and metadata for each training run are stored as specific *model versions* within this container.

### Key Details
* **Usage Privileges:** The basic permissions model for registered models is the same as it is for functions (requiring `USE CATALOG` and `USE SCHEMA` on the parent objects).
* **Specific Model Privileges:** 
  * `CREATE MODEL`: Required on the parent schema or catalog to create a new model container.
  * `CREATE MODEL VERSION`: Allows a user to register new versions of an existing model without granting them the ability to execute, modify, or add tags to the model itself.
  * `APPLY TAG`: Allows a user to add and edit tags on a model and its specific versions.
* **Inheritance:** Privileges like `CREATE MODEL` granted at the catalog level automatically allow the user to create models in any schema within that catalog.

## Service

Within a schema, a **service** is a securable object that represents a governed, invocable AI asset. Services allow you to manage access to your AI traffic using the exact same privilege system you use for your data.

### Types of Services (Currently in Beta)
* **Model services:** Expose LLM endpoints that you can invoke and share across workspaces.
* **Model provider services:** Store connection details and encrypted credentials for external model providers, allowing teams to query external models without directly accessing the underlying credentials.
* **MCP services:** Register MCP servers so you can govern which tools your agents are allowed to use.
* **Agent services:** Register AI agents so you can control who is allowed to invoke them.

### Key Details
* **Usage Privileges:** To invoke a service, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema, in addition to the `EXECUTE` privilege on the service itself.
* **Creation Requirements:** Creating a new service requires the `CREATE SERVICE` privilege on the parent schema or catalog.
* **The `EXECUTE` Privilege:** Granting `EXECUTE` on a service allows a user to invoke it. This works the same way as the `EXECUTE` privilege used for functions and models.

## Secret

Within a schema, a **secret** is a securable object used to securely store sensitive values, such as credentials or API tokens. Secrets allow you to govern access to these sensitive values in Unity Catalog and use them safely in your code or UDFs without ever exposing the actual text of the value. They follow the standard three-level namespace (`catalog.schema.secret`).

### Key Details
* **Usage Privileges:** To use a secret, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema, in addition to a specific secret-level privilege.
* **Secret Privileges:**
  * `CREATE SECRET`: Allows a user to create new secrets within a schema.
  * `READ SECRET`: Allows a user to retrieve the actual underlying value of the secret.
  * `WRITE SECRET`: Allows a user to update or change the secret's value.
  * `REFERENCE SECRET`: Allows a user to see that the secret exists and view its metadata, *without* granting access to the actual sensitive value.
  * `APPLY TAG`: Allows a user to add and edit tags on a secret.
* **Inheritance:** Secret privileges granted at the schema or catalog level automatically apply to all current and future secrets within that schema or catalog.

## Feature

Within a schema, a **feature** is a securable object that represents the stored definition of a machine learning feature. This includes its source data, computation logic, and the time windows used to compute it. It acts as a single, governed source of truth that can be reused across training, materialization, and serving, while Unity Catalog automatically tracks its lineage back to the source data and models that use it.

### Key Details
* **Metadata Only:** Defining a feature is strictly a metadata operation. No actual feature data is computed until the feature is materialized.
* **Usage Privileges:** To read a feature, a user must have `USE CATALOG` on the parent catalog and `USE SCHEMA` on the parent schema (usage privileges), in addition to the `READ FEATURE` privilege on the feature itself.
* **Creation Requirements:** To create a new feature in a schema, a user must have the `CREATE FEATURE` privilege on the schema, along with the standard usage privileges on the parent catalog and schema.
* **Read Access Limitation:** The `READ FEATURE` privilege only lets a user read the feature's metadata. It *does not* grant access to the actual feature data in the source or materialized output tables. To read that data for model inference or training, the user must also have the `SELECT` privilege on those applicable tables.
* **Inheritance:** The `READ FEATURE` privilege granted at the schema or catalog level automatically applies to all current and future features in that schema or catalog.

## Storage Credential

Within a metastore, a **storage credential** is a securable object that securely stores the authentication information required to access a specific path in your cloud storage. The authentication method depends on your cloud provider (for example, a service principal on Azure, an IAM role on AWS, or a service account on GCP).

### Key Details
* **Primary Use:** Storage credentials are most commonly used as a foundational building block for **external locations** (which pair a storage credential with a specific cloud storage path). They can also be used directly to create external tables.
* **Creation Requirements:** To create a new storage credential, a user must have the `CREATE STORAGE CREDENTIAL` privilege directly on the Unity Catalog metastore.

## External Location

Within a metastore, an **external location** is a securable object that pairs a storage credential with a specific cloud storage path. It governs access to that specific path in your cloud storage.

### Key Details
* **Creation Requirements:** To create an external location, a user needs the `CREATE EXTERNAL LOCATION` privilege on the Unity Catalog metastore.
* **Read and Write Access:** Users need the `READ FILES` privilege to read files directly from the storage path, and the `WRITE FILES` privilege to write files.
* **Best Practice:** Databricks recommends managing cloud storage access through volumes (using `READ VOLUME` and `WRITE VOLUME` privileges) rather than granting `READ FILES` and `WRITE FILES` directly on external locations.

## External Metadata

Within a metastore, an **external metadata** object is a securable object used to define custom data lineage relationships for systems that operate outside of Unity Catalog's native lineage tracking.

### Key Details
* **Creation Requirements:** To create an external metadata object, a user needs the `CREATE EXTERNAL METADATA` privilege on the Unity Catalog metastore.
* **Modification Privileges:** To add or modify lineage relationships on this object, a user needs the `MODIFY` privilege on the external metadata object itself, plus the appropriate privileges on any Unity Catalog objects referenced in the relationship.

## Service Credential

Within a metastore, a **service credential** is a securable object that stores authentication information for accessing external cloud services. This differs from storage credentials, which are specifically for governing access to cloud storage.

### Key Details
* **Creation Requirements:** To create a service credential, a user needs the `CREATE SERVICE CREDENTIAL` privilege on the Unity Catalog metastore.
* **Usage Privileges:** The `ACCESS` privilege allows a user to use the service credential to access an external service.
* **Connection Creation:** Granting `CREATE CONNECTION` on a service credential (combined with `CREATE CONNECTION` on the metastore) allows a user to create a connection to an external database using that credential.

## Connection

Within a metastore, a **connection** is a securable object that securely stores the endpoint and credentials needed to access an external system. Connections are used for scenarios like query federation, catalog federation, managed ingestion, JDBC access, and HTTP services.

### Key Details
* **Creation Requirements:** To create a connection, a user needs the `CREATE CONNECTION` privilege on the Unity Catalog metastore. If the connection relies on a service credential, the user also needs the `CREATE CONNECTION` privilege on that specific service credential.
* **Usage Privilege (`USE CONNECTION`):** The `USE CONNECTION` privilege allows a user to list the connection, view its details, and use the connection for its supported scenario (like querying the external database).

## Share

Within a metastore, a **share** is a securable object in OpenSharing that represents a logical grouping of data assets (like tables, views, and volumes) that you intend to share with external recipients. 

### Key Details
* **Creation Requirements:** To create a share, a user needs the `CREATE SHARE` privilege directly on the Unity Catalog metastore.
* **Access Control:** The `SELECT` privilege on a share is granted directly to a *recipient* object (representing the external organization), not to individual users. This allows that recipient to read the assets contained in the share.

## Provider

Within a metastore, a **provider** is a securable object in OpenSharing that represents an external organization that has shared data with your organization. Provider objects are created in the *recipient's* Unity Catalog metastore.

### Key Details
* **Creation Requirements:** To create a provider, a user needs the `CREATE PROVIDER` privilege on the Unity Catalog metastore.
* **Usage Privilege (`USE PROVIDER`):** This privilege allows a user to view all providers and their available shares. Combined with the `CREATE CATALOG` privilege, it allows a user to mount a shared catalog without needing to be a metastore admin.

## Recipient

Within a metastore, a **recipient** is a securable object in OpenSharing that represents an external organization or group of users that a provider shares data with. Recipient objects are created in the *provider's* Unity Catalog metastore.

### Key Details
* **Creation Requirements:** To create a recipient, a user needs the `CREATE RECIPIENT` privilege on the Unity Catalog metastore.
* **No Direct Privileges:** You do not grant privileges directly on a recipient object itself. Instead, access to shared data is controlled by granting the `SELECT` privilege on a specific **share** to that recipient.

## Clean Room

Within a metastore, a **clean room** is a securable object that provides a secure, trusted environment for collaborating with other organizations on shared data, without either party exposing their underlying raw data to the other.

### Key Details
* **Creation Requirements:** To create a clean room, a user needs the `CREATE CLEAN ROOM` privilege on the Unity Catalog metastore.
* **Execution Access:** The `EXECUTE CLEAN ROOM TASK` privilege allows a user to run notebooks inside the clean room and view the clean room's details.
* **Modification Access:** The `MODIFY CLEAN ROOM` privilege allows a user to update the clean room's configuration, which includes adding or removing data assets, notebooks, and comments.

## Flows in Unity Catalog

A **flow** (currently in Beta) is a query that incrementally loads and processes data—either as a batch or as a stream—into a target. In Unity Catalog, each pipeline flow is treated as a securable object that lives inside a schema, right alongside the tables it writes to. It follows the standard three-level namespace (`catalog.schema.flow_name`).

### Key Characteristics
* **Pipeline:** A flow belongs to the Lakeflow pipeline that defines it, and it is updated directly by that pipeline. The pipeline also manages the flow's ownership.
* **Target:** A flow writes to a single target streaming table or materialized view. (Note: A single target table can receive data from more than one flow).
* **Flow type:** A flow records exactly how it processes data, using methods such as *append*, *Auto CDC*, or *replace where*.

### Privileges and Access Control
Because flows are securable objects, you can grant fine-grained permissions on them without having to grant broader access to the entire pipeline or the target table. 

* **Usage Privileges:** Just like other objects, accessing a flow requires the `USE CATALOG` privilege on the parent catalog and the `USE SCHEMA` privilege on the parent schema.
* **Specific Flow Privileges:**
  * `CREATE FLOW`: Allows a user to create a flow in the schema. (Note: Adding a flow to an existing target table also requires the `MODIFY` privilege on that table).
  * `READ FLOW`: Allows a user to describe the flow and read its definition and current state.
  * `REFRESH FLOW`: Allows a user to refresh the flow.
  * `WRITE FLOW`: Allows a user to reset or rewind the flow's streaming checkpoint.
  * `MANAGE`: Allows a user to view and manage privileges on the flow, rename it, or drop it.
* **Inheritance:** Flow privileges granted at the catalog or schema level automatically inherit down to all current and future flows within that catalog or schema.

## Managed versus External Assets

In Unity Catalog, every securable object you register has its access, auditing, and lineage centrally governed. However, for data assets like **tables** and **volumes**, there is a crucial distinction based on who controls the actual underlying data files and their storage lifecycle.

### Summary of Differences
* **Managed Assets:** Unity Catalog controls *both* the governance AND the underlying file storage lifecycle (including where files are stored, how they are optimized, and when they are deleted). When you drop a managed asset, the underlying data files are permanently deleted (after a default 7-day recovery period).
* **External Assets:** Unity Catalog controls *only* the governance. You (or an external system) control the file storage lifecycle. When you drop an external asset, the Unity Catalog metadata is removed, but your underlying data files safely remain in place.
* **Data Ownership:** For both managed and external assets, you retain full ownership of your data, and the data files always remain in your own cloud account.

### Managed and External Tables
* **Managed tables:** Unity Catalog determines the storage location (using the managed storage location defined at the schema, catalog, or metastore level). They use Delta or Apache Iceberg formats. Both Databricks and external engines can access them via open APIs, preventing vendor lock-in.
* **External tables:** You manually specify the storage location for the underlying data files. They support multiple formats, including Delta, CSV, JSON, Avro, Parquet, and ORC.

### Managed and External Volumes
* **Managed volumes:** Unity Catalog automatically determines the storage location within your cloud account's designated managed storage location.
* **External volumes:** You specify the storage location, which must be a path already covered by a Unity Catalog external location.

### Uses of the Word "Manage"
The word "manage" is used in a few different contexts across Unity Catalog:
* **"Managed by Unity Catalog":** Means Unity Catalog governs access, auditing, and lineage for the object (this applies to *all* registered objects, including external ones).
* **Managed table/volume:** Means Unity Catalog controls the underlying storage location and data lifecycle.
* **Managed storage location:** The designated path in your cloud account where Unity Catalog stores data files for managed tables and volumes.
* **`MANAGE` privilege:** A specific permission that allows a user to assign/revoke privileges, transfer ownership, and delete an object without actually being its owner.

## Object Storage Lifecycle in Unity Catalog

When you delete a securable object in Unity Catalog (e.g., using `DROP`), the resulting behavior for the underlying data files, storage billing, and recovery options depends entirely on the object's type and its storage type.

### Storage Type Behaviors
* **Managed tables and volumes:** Unity Catalog completely controls the storage location and data file lifecycle. When deleted, Unity Catalog permanently deletes the underlying data files through a multi-phase lifecycle. This applies to both Azure Databricks default storage and customer-provided managed storage.
* **External tables and volumes:** You control the storage location and lifecycle. When deleted, Unity Catalog only removes the metadata from the metastore. The underlying data files safely remain in your cloud storage, and you must delete them manually from your cloud provider if desired.
* **Foreign and federated catalogs:** Data lives in an external data source (like Lakehouse Federation). Unity Catalog only holds the connection metadata. Deleting these only removes the connection metadata; the source system data is completely unaffected.
* **Other objects (catalogs, schemas, views, models, functions):** Deleting these only removes the metadata, as there are no direct data files for Unity Catalog to manage. (Note: Dropping a catalog or schema with `CASCADE` will recursively remove the contained tables and volumes according to their managed/external behavior).

### Lifecycle of Managed Data After Deletion
When a managed table or volume is deleted, the data files are not deleted immediately. They go through a two-phase lifecycle:
* **Phase 1: Recovery Window (7 Days):** For 7 days after the delete, Unity Catalog retains the soft-deleted data. You can recover the object during this time. Storage billing continues during this phase.
* **Phase 2: Purge (After 7 Days):** Once the 7-day window ends, the object can no longer be recovered. An asynchronous purge process permanently deletes the data files. (Note: The purge is not instantaneous and can take anywhere from 7 up to 30 days after the initial delete to fully reclaim cloud storage capacity).

### Storage Billing After Deletion
* **Azure Databricks default storage:** Azure Databricks bills for storage *only* during the 7-day recovery window. Billing stops entirely once the recovery window passes.
* **Customer-provided managed storage:** Your cloud provider bills you directly. Even after Unity Catalog purges the files, you may still incur charges depending on your specific cloud bucket's object versioning, soft-delete, and lifecycle policies.
* **External storage:** Your cloud provider continues to bill you indefinitely according to your bucket's policies, as Unity Catalog does not delete these files.

### Recovering Deleted Objects
* **Tables, materialized views, streaming tables:** Can be recovered using the `UNDROP` command within the 7-day recovery period. (Note: Materialized views and streaming tables must have been created from an ETL pipeline that still exists).
* **Volumes:** Can be recovered using the `UNDROP VOLUME` command within 7 days, provided the parent schema and catalog still exist.
* **Catalogs, schemas, views, functions, models:** Cannot be recovered after they are deleted. (Best Practice: Use the default `RESTRICT` option on `DROP CATALOG` and `DROP SCHEMA` to prevent accidental recursive deletion of non-empty containers).

## Unity Catalog Requirements and Limitations

To successfully use Unity Catalog, your Azure Databricks environment must meet specific compute, formatting, and naming requirements. There are also several known limitations to keep in mind as you design your data governance strategy.

### Compute Requirements
* **Databricks Runtime (DBR):** Unity Catalog requires clusters running Databricks Runtime 11.3 LTS or higher. 
* **SQL Warehouses:** Unity Catalog is supported by default on all SQL warehouse compute versions.
* **Access Modes:** To access data in Unity Catalog, your cluster must be configured with a compatible access mode (either **standard** or **dedicated**). If a cluster is not using one of these modes, it cannot access Unity Catalog data because Unity Catalog is secure by default.

### File Format Support
The table formats you can use depend on whether the table is managed or external:
* **Managed tables:** Must use the **Delta** or **Iceberg** format.
* **External tables:** Offer more flexibility and can use **Delta, CSV, JSON, Avro, Parquet, ORC,** or **Text**.

### Naming Requirements for Securable Objects
When naming objects (like catalogs, schemas, or tables) in Unity Catalog, you must follow these rules:
* **Length:** Names cannot exceed 255 characters.
* **Prohibited Characters:** You cannot use periods (`.`), spaces (` `), forward slashes (`/`), ASCII control characters, or the DELETE character.
* **Case Sensitivity:** Unity Catalog stores all object names in lowercase. Queries against these tables are case-insensitive.
* **Escaping:** If you reference names containing special characters (like hyphens) in SQL, you must wrap them in backticks (`` ` ``). 

### Key Limitations
While Unity Catalog is constantly updated, there are a few important limitations to remember:
* **Workspace vs. Account Groups:** You cannot use old workspace-level groups in `GRANT` statements. To grant privileges across workspaces, you must use **account-level groups**.
* **Cloning Restrictions:** You can shallow clone a managed table to another managed table, and an external table to another external table. However, you **cannot** shallow clone across types (e.g., you cannot clone a managed table to an external table).
* **Bucketing:** Creating bucketed tables is not supported in Unity Catalog.
* **Resource Quotas:** Unity Catalog enforces resource limits (quotas) on all securable objects. If you expect massive scale, you need to monitor these limits and work with your Databricks team if you need increases.